In [ ]:
# Only use when connecting to Colab GPU from VSCode extension
# Sets up Colab environment --> Should take ~3 minutes

from google.colab import drive
import os
drive.mount('/content/drive')

!mkdir -p ~/.ssh
!cp /content/drive/MyDrive/colab_keys/id_ed25519 ~/.ssh/
!cp /content/drive/MyDrive/colab_keys/id_ed25519.pub ~/.ssh/

!chmod 600 ~/.ssh/id_ed25519
!ssh-keyscan github.com >> ~/.ssh/known_hosts

if not os.path.exists("CMPM118-Winter2026"):
    !git clone git@github.com:Tardigrades3/CMPM118-Winter2026.git
    
%cd CMPM118-Winter2026
!git checkout aran-dev
# !pip install -e .
!python download_ninapro.py

In [2]:
import torch
from torch import nn
from tqdm import tqdm
import copy
from torch.utils.data import Subset
import torch.nn.functional as F

import sys, os
sys.path.append(os.path.abspath("../"))
import preprocessing
import numpy as np



In [46]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using {device} device")

# Base MNIST NN

class NinaProClassificationModel(nn.Module):
  def __init__(self):
    super().__init__()
    self.conv1 = nn.Conv1d(10, 64, kernel_size=5, stride=2)
    self.relu1 = nn.ReLU()
    self.pool1 = nn.MaxPool1d(kernel_size=2)
    self.drop1 = nn.Dropout(0.1)

    self.conv2 = nn.Conv1d(64, 128, kernel_size=5, stride=2)
    self.relu2 = nn.ReLU()
    self.pool2 = nn.MaxPool1d(kernel_size=2)
    self.drop2 = nn.Dropout(0.1)

    self.flatten = nn.Flatten()
    
    self.dense1 = nn.LazyLinear(256)
    self.relu3 = nn.ReLU()
    self.drop3 = nn.Dropout(0.4)
    self.dense2 = nn.Linear(256, 128)
    self.relu4 = nn.ReLU()
    self.drop4 = nn.Dropout(0.3)
    self.dense3 = nn.Linear(128, 13)

  def forward(self, x):
    out = self.relu1(self.conv1(x))
    out = self.pool1(out)
    out = self.drop1(out)
    out = self.relu2(self.conv2(out))
    out = self.pool2(out)
    out = self.drop2(out)
    out = self.flatten(out)
    out = self.relu3(self.dense1(out))
    out = self.drop3(out)
    out = self.relu4(self.dense2(out))
    out = self.drop4(out)
    out = self.dense3(out)
    return out

class NinaProDataset(torch.utils.data.Dataset):
  def __init__(self, x, y):
    self.X = torch.tensor(x, dtype=torch.float32)
    self.y = torch.tensor(y, dtype=torch.long)

  def __len__(self):
    return len(self.X)
  
  def __getitem__(self, idx):
    return self.X[idx], self.y[idx]

# Base NN


model = NinaProClassificationModel().to(device)

optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

# path should be ../NinaProData for local machine
x_train, y_train, x_test, y_test, class_weights = preprocessing.multi_preprocess(exercise_number=1, path="./NinaProData")
batch_size = 100

# Since resting is a massive portion of our dataset, remove all resting labels for now.
def drop_rest_label(x_train, y_train, x_test, y_test):
    mask = y_train != 0
    x_train = x_train[mask]
    y_train = y_train[mask]

    mask = y_test != 0
    x_test = x_test[mask]
    y_test = y_test[mask]

    y_train -= 1
    y_test -= 1
    return x_train, y_train, x_test, y_test

x_train, y_train, x_test, y_test = drop_rest_label(x_train, y_train, x_test, y_test)

train_data = NinaProDataset(x_train, y_train)
test_data = NinaProDataset(x_test, y_test)
batched_train = torch.utils.data.DataLoader(train_data, batch_size=batch_size, shuffle=True)
batched_test = torch.utils.data.DataLoader(test_data, batch_size=batch_size, shuffle=False)


loss_func = nn.CrossEntropyLoss()

Using cuda device
subject #[[1]]
exercise #[[1]]
0.0     351
5.0      58
3.0      53
1.0      50
8.0      44
6.0      43
12.0     42
4.0      41
10.0     39
2.0      37
7.0      36
9.0      35
11.0     35
Name: count, dtype: int64
0.0     908
3.0      30
1.0      26
10.0     25
12.0     23
5.0      22
4.0      18
9.0      17
6.0      17
7.0      17
2.0      16
8.0      16
11.0     15
Name: count, dtype: int64
subject #[[2]]
exercise #[[1]]
0.0     291
5.0      55
7.0      54
3.0      53
8.0      50
11.0     50
4.0      49
9.0      48
2.0      48
6.0      46
12.0     42
10.0     39
1.0      37
Name: count, dtype: int64
0.0     880
8.0      28
3.0      26
5.0      24
1.0      24
4.0      23
7.0      22
10.0     21
9.0      21
2.0      20
6.0      20
11.0     18
12.0     18
Name: count, dtype: int64
subject #[[3]]
exercise #[[1]]
0.0     339
2.0      54
3.0      52
6.0      47
5.0      47
12.0     46
1.0      44
7.0      43
4.0      42
8.0      40
9.0      39
10.0     35
11.0     34
Name:

In [47]:
# TODO: Ideas to improve model
# Filter out resting position(Tyler's job)
# Integrate class weights somehow to fix imbalances

print("Starting Training")
epochs = 50
model.train()

for epoch in range(epochs):
    pbar = tqdm(batched_train, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
    for emg, label in pbar:
        emg = emg.to(device)
        label = label.to(device)
        optimizer.zero_grad()
        logits = model(emg)
        loss = loss_func(logits, label)
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=loss.item())
        

Starting Training


In [57]:
# Accuracy:
def find_accuracy(model, batched_test):
    model.eval()
    total_loss = 0
    total = 0
    correct = 0
    with torch.no_grad():
        correct = 0
        for emg, labels in batched_test:
            emg = emg.to(device)
            labels = labels.to(device)
            logits = model(emg)
            loss = loss_func(logits, labels)

            total_loss += loss.item() * emg.size(0)
            preds = logits.argmax(dim=1)
            correct += (preds == labels).sum().item()
            total += emg.size(0)
    print(f"Average Loss: {total_loss/correct}, Accuracy: {correct/total}")
find_accuracy(model, batched_test)

Average Loss: 1.9213414909506086, Accuracy: 0.7276791416528292


In [ ]:
!pip install torchmetrics

In [ ]:
from torchmetrics.classification import MulticlassAccuracy

num_classes = 12
per_class_acc = MulticlassAccuracy(
    num_classes=num_classes,
    average=None
).to(device)

model.eval()

with torch.no_grad():
    for emg, label in batched_test:
        emg = emg.to(device).float()
        label = label.to(device)
        logits = model(emg)
        preds = logits.argmax(dim=1)
        per_class_acc.update(preds, label)

accs = per_class_acc.compute()


In [53]:
for i, a in enumerate(accs):
    print(f"Class {i}: {a.item():.3f}")

Class 0: 0.993
Class 1: 0.985
Class 2: 0.989
Class 3: 0.997
Class 4: 0.979
Class 5: 0.983
Class 6: 0.983
Class 7: 0.989
Class 8: 0.978
Class 9: 0.982
Class 10: 0.959
Class 11: 0.974


In [16]:
def expand_classifier(model, new_size, device):
  old_fc = model.dense3
  in_features = old_fc.in_features
  old_out = old_fc.out_features
  newlayer = nn.Linear(in_features, old_out + new_size).to(device)

  with torch.no_grad():
    newlayer.weight[:old_out].copy_(old_fc.weight)
    newlayer.bias[:old_out].copy_(old_fc.bias)
  model.dense3 = newlayer
  return model

In [ ]:
# CIL / KWF

x2_train, y2_train, x2_test, y2_test, class_weights_2 = preprocessing.multi_preprocess(2, "./NinaProData")

# Since resting is a massive portion of our dataset, remove all resting labels for now.
x2_train, y2_train, x2_test, y2_test = drop_rest_label(x2_train, y2_train, x2_test, y2_test)


train2_data = NinaProDataset(x2_train, y2_train)
test2_data = NinaProDataset(x2_test, y2_test)

batch_size = 100
train2_loader = torch.utils.data.DataLoader(train2_data, batch_size=batch_size, shuffle=True)
test2_loader = torch.utils.data.DataLoader(test2_data, batch_size=batch_size, shuffle=False)

num_new_classes = len(np.unique(y2_train))

student = copy.deepcopy(model).to(device)
student = expand_classifier(student, new_size=num_new_classes, device=device)
student = student.to(device)



In [ ]:
student = student.to(device)
student.train()

optimizer = torch.optim.Adam(student.parameters(), lr = 0.001)
epochs = 1000
T = 4
alpha_kd = .9

for epoch in range(epochs):
    if epoch%10 == 0 or epoch < 5: find_accuracy(student, test2_loader)
    pbar = tqdm(train2_loader, desc=f"Epoch {epoch+1}/{epochs}", leave=False)
    for emg, labels in pbar:
        emg = emg.to(device)
        labels = labels.to(device)
        logits_stud_new = student(emg)
        loss_stud_new = loss_func(logits_stud_new[:, 13:], labels)
        
        with torch.no_grad():
            logits_teach = model(emg)
            loss_teach = F.softmax(logits_teach/T, dim = 1)
        loss_stud_old = F.log_softmax(logits_stud_new[:, :13]/T, dim=1)
        loss_old = F.kl_div(loss_stud_old, loss_teach, reduction = "batchmean") * (T**2)
        
        loss = (1-alpha_kd) * loss_stud_new + alpha_kd * loss_old
        optimizer.zero_grad()
                
        loss.backward()
        optimizer.step()
        pbar.set_postfix(loss=loss.item())
        


In [ ]:
print(f"Accuracy for old task")
find_accuracy(student, batched_test)
print(f"Accuracy for new task")
find_accuracy(student, test2_loader)